In [1]:
# imports 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import random
import os, sys
sys.path.append(os.getcwd())
import glob
import numpy as np
import copy

from data_processing import build_pair_df, prepare_data


# device setup 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # BUT I NEED TO CONNECT TO AWS? 
print("Device:", device)

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [2]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
from data_processing import build_pair_df, prepare_data

# STEP 1: call data processing functions
pair_df = build_pair_df(
        base_path=".",
        sample_n=None, # all data
        image_dir="images",
        random_state=42
)

train_loader, val_loader, test_loader = prepare_data(
    pair_df,
    seed=42,
    batch_size=64,
)

62916 20972 20972
158991 17765 17465


In [ ]:
# STEP 2: load ResNet architecture
def get_resnet_backbone():
    model = models.resnet50(weights="IMAGENET1K_V1")

    modules = list(model.children())[:-2]  # keep only conv blocks, output = [B, 2048, 7, 7]
    backbone = nn.Sequential(*modules)

    # extract features before the final classifier 
    features = nn.Sequential(*list(backbone.children())[:-1]) 

    # freeze all layers
    for param in backbone.parameters():
        param.requires_grad = False

    # unfreeze last block
    for name, param in backbone.named_parameters():
        if "layer4" in name:    # last block in ResNet50
            param.requires_grad = True

    # unfreeze BatchNorm layers in layer4
    for m in features.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

    return backbone

In [ ]:
# STEP 3: SS-CNN model (two-branch network)
class SSCNN(nn.Module):
    def __init__(self, backbone, feat_dim=512): 
        super().__init__() # initialize parent class
        
        self.backbone = backbone # pretrained backbone
        
        # white box: fusion classifier (3 conv layers + 2-unit classifier)
        self.fusion_head = nn.Sequential(
                    nn.Conv2d(feat_dim*2, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv2d(512, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv2d(512, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                )

        self.fusion_fc = nn.Linear(512 * 7 * 7, 2)

    # define a forward pass 
    def forward_once(self, x):
        # get info for classification 
        conv_feat = self.backbone(x)  # [B, 2048, 7, 7]
        return conv_feat


    # define running a forward pass on both images 
    def forward(self, imgA, imgB):
        convA = self.forward_once(imgA)  # [B, 2048, 7, 7]
        convB = self.forward_once(imgB)  # [B, 2048, 7, 7]

        # Concatenate at the channel dimension for convolutional fusion
        fusion = torch.cat([convA, convB], dim=1)  # [B, 4096, 7, 7]
        
        # Pass through convolutional fusion layers
        fusion_feat = self.fusion_head(fusion)  # [B, 512, 7, 7]
        
        # Flatten for the final fully connected layer
        fusion_flat = torch.flatten(fusion_feat, 1)  # [B, 512*7*7]
        
        # Final classification
        logits = self.fusion_fc(fusion_flat)  # [B, 2]

        return logits

In [ ]:
# STEP 4: loss function
criterion = nn.CrossEntropyLoss()

def classification_loss(logits, label): 
    return criterion(logits, label)

In [ ]:
# STEP 4b: Evaluation loss (no training)
def eval_loss(dataloader, model, loss_fn):
    model.eval()  # Set to evaluation mode
    total_loss = 0

    with torch.no_grad():  # no gradient computation
        for imgA, imgB, label, y in dataloader:
            imgA = imgA.to(device)
            imgB = imgB.to(device)
            label = label.to(device).long()

            logits = model(imgA, imgB)
            loss = loss_fn(logits, label)

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# STEP 5: training loop 
def train_one_epoch(dataloader, model, optimizer):
    model.train() # set model to training mode
    
    total_loss = 0 # initialize total loss

    for imgA, imgB, label, y in dataloader: 
        imgA = imgA.to(device) # move to device
        imgB = imgB.to(device) # move to device
        label = label.to(device).long() # move to device

        logits = model(imgA, imgB) # forward pass
        

        loss = classification_loss(logits, label) # compute classification loss

        optimizer.zero_grad() # zero gradients
        loss.backward() # backpropagation
        optimizer.step() # update weights

        total_loss += loss.item() # accumulate loss

    return total_loss / len(dataloader) # average loss

In [ ]:
# STEP 7: accuracy, precision, & recall calculation 

def accuracy(dataloader, model):
    model.eval()

    total_correct = 0
    total = 0

    with torch.no_grad():
        for imgA, imgB, label, y in dataloader:  
            imgA = imgA.to(device)
            imgB = imgB.to(device)
            label = label.to(device)  

            logits = model(imgA, imgB)
            preds = logits.argmax(dim=1)  

            # accuracy
            total_correct += (preds == label).sum().item()
            total += label.size(0)


    accuracy = total_correct / total

    return accuracy 

In [ ]:
# STEP 7b: accuracy, precision, & recall calculation

def accuracy_precision_recall(test_dataloader, model):
    TP = 0  # True Positives
    FP = 0  # False Positives
    FN = 0  # False Negatives
    total_correct = 0
    total = 0

    with torch.no_grad():
        for imgA, imgB, label, y in test_dataloader:  # ← Now unpacking 4 values
            imgA = imgA.to(device)
            imgB = imgB.to(device)
            label = label.to(device)  # shape [B]

            logits = model(imgA, imgB)
            preds = logits.argmax(dim=1)  # shape [B]

            # accuracy
            total_correct += (preds == label).sum().item()
            total += label.size(0)

            # precision/recall components
            TP += ((preds == 1) & (label == 1)).sum().item()
            FP += ((preds == 1) & (label == 0)).sum().item()
            FN += ((preds == 0) & (label == 1)).sum().item()

    accuracy = total_correct / total
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    return accuracy, precision, recall, TP, FP, FN

In [ ]:
search_results = []

# Define hyperparameter ranges
lr = .0005
batch_size = 64

# Re-Initialize Model & Optimizer
backbone = get_resnet_backbone()
model = SSCNN(backbone).to(device)  
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Setup Training Config
num_epochs = 20       
patience = 15          
best_val_acc = 0.0     
patience_counter = 0   
best_model_wts = copy.deepcopy(model.state_dict())

# Local tracking 
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

# --- START TRAINING EPOCHS ---
for epoch in range(num_epochs):
    train_loss = train_one_epoch(train_loader, model, optimizer)
    val_loss = eval_loss(val_loader, model, classification_loss)

    train_accuracy = accuracy(train_loader, model)
    val_accuracy = accuracy(val_loader, model)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)
    
    print(f"  Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f}")

    # Early Stopping Logic
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        patience_counter = 0 
        best_model_wts = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

# Record Results for this Trial
search_results.append({
    'hyperparameters': {'lr': lr, 'batch_size': batch_size},
    'best_val_acc': best_val_acc,
    'history_acc': val_accuracies,
    'losses': train_losses
})

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")


In [ ]:
# STEP 9: Test Set Evaluation

# load best model weights
model.load_state_dict(best_model_wts)

# calculate test metrics
test_accuracy, test_precision, test_recall, TP_test, FP_test, FN_test = accuracy_precision_recall(test_loader, model)

# calculate true negatives for completeness
TN = 0

model.eval()
with torch.no_grad():
    for imgA, imgB, label, y in test_loader:
        imgA = imgA.to(device)
        imgB = imgB.to(device)
        label = label.to(device)

        logits = model(imgA, imgB)
        preds = logits.argmax(dim=1)

        TN += ((preds == 0) & (label == 0)).sum().item()

# print test results
print(f"\nTest Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Positives (TP):  {TP_test}")
print(f"  False Positives (FP): {FP_test}")
print(f"  False Negatives (FN): {FN_test}")
print(f"  True Negatives (TN):  {TN}")
print(f"\nTest Set Size: {TP_test + FP_test + FN_test + TN}")
print("="*50)